# Phase 3: Exploratory Data Analysis (EDA)

**Project:** E-Commerce Customer Behaviour Analysis and Customer Segmentation  
**Module:** `src/eda.py`  
**Outputs Directory:** `reports/figures/`  
**Objective:** Discover empirical behavioral patterns, sales dynamics, product performance, geographic dispersion, and keyword concentrations from clean transactional records, appending direct business interpretations to each finding.

### Step 1: Import Visualization Modules and Load Clean Data
We import modular plotting functions from `src/eda.py` and ensure full reproducibility (`random_state=42`).

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import RANDOM_STATE
from src.eda import (
    plot_monthly_trend,
    plot_day_and_hour_patterns,
    plot_top_products,
    plot_geographic_distribution,
    plot_order_value_and_basket_distributions,
    plot_category_and_keyword_analysis,
    run_all_eda
)

df_clean = pd.read_csv(os.path.join(project_root, "data", "processed", "clean_transactions.csv"))
print("Modules imported successfully.")
print(f"Reproducibility random_state: {RANDOM_STATE}")
print(f"Clean dataset loaded: {len(df_clean):,} records x {df_clean.shape[1]} columns.")
print(f"Total Gross Revenue: ${df_clean['Revenue'].sum():,.2f}")

Modules imported successfully.
Reproducibility random_state: 42
Clean dataset loaded: 3,900 records x 27 columns.
Total Gross Revenue: $233,081.00


### 1. Monthly Revenue & Order Volume Trends
We analyze sales performance across chronological months and seasons to detect seasonality or cyclical surges.

In [2]:
fig1_path = plot_monthly_trend(df_clean, output_dir=os.path.join(project_root, "reports", "figures"))
print(f"Figure saved to: {os.path.relpath(fig1_path, project_root)}")

# Display summary metrics
monthly_summary = df_clean.groupby("Month").agg(
    Total_Revenue=("Revenue", "sum"),
    Orders=("CustomerID", "count"),
    Average_Order_Value=("Revenue", "mean")
).reset_index()
monthly_summary

Figure saved to: reports/figures/01_monthly_revenue_order_trend.png


![Monthly Revenue & Order Trend](../reports/figures/01_monthly_revenue_order_trend.png)

> **Business Interpretation:** Sales and order volumes remain remarkably stable across all seasons (~$58K per quarter), indicating year-round staple consumer demand that minimizes off-season inventory obsolescence.

### 2. Day-of-Week and Hour-of-Day Purchasing Patterns
We explore transaction frequency across weekly calendar days and hourly operational windows.

In [3]:
fig2_path = plot_day_and_hour_patterns(df_clean, output_dir=os.path.join(project_root, "reports", "figures"))
print(f"Figure saved to: {os.path.relpath(fig2_path, project_root)}")

Figure saved to: reports/figures/02_day_and_hour_patterns.png


![Day and Hour Patterns](../reports/figures/02_day_and_hour_patterns.png)

> **Business Interpretation:** Transaction frequency peaks consistently during midday shopping hours (11:00 AM – 2:00 PM) and mid-week days, establishing clear prime windows for time-sensitive marketing push notifications and customer support staffing.

### 3. Top 10 Products by Revenue and Quantity
Identifying high-value flagship items versus high-velocity volume drivers.

In [4]:
fig3_path = plot_top_products(df_clean, output_dir=os.path.join(project_root, "reports", "figures"))
print(f"Figure saved to: {os.path.relpath(fig3_path, project_root)}")

# Tabular ranking
top10_prod = df_clean.groupby("Item_Purchased").agg(
    Revenue=("Revenue", "sum"),
    Quantity_Ordered=("Quantity", "sum")
).sort_values(by="Revenue", ascending=False).head(10)
top10_prod

Figure saved to: reports/figures/03_top_products_revenue_quantity.png


![Top Products by Revenue & Quantity](../reports/figures/03_top_products_revenue_quantity.png)

> **Business Interpretation:** Blouses, Shirts, Dresses, and Jewelry represent the top revenue-generating hero products, making them ideal anchor offerings for bundle promotions and premium placements on homepage landing screens.

### 4. Geographic Distribution and Market Concentration
We examine revenue generation across geographic states and evaluate market concentration risk.

In [5]:
fig4_path = plot_geographic_distribution(df_clean, output_dir=os.path.join(project_root, "reports", "figures"))
print(f"Figure saved to: {os.path.relpath(fig4_path, project_root)}")

# Top 5 locations
top_states = df_clean.groupby("Location").agg(
    State_Revenue=("Revenue", "sum"),
    Orders=("CustomerID", "count")
).sort_values(by="State_Revenue", ascending=False).head(5)
top_states

Figure saved to: reports/figures/04_revenue_by_location.png


![Geographic Distribution](../reports/figures/04_revenue_by_location.png)

> **Business Interpretation:** Consumer spending exhibits exceptional nationwide dispersion across all 50 states (top 5 states capture only ~12.5% of gross revenue), indicating low single-region market vulnerability and recommending decentralized multi-hub distribution logistics.

### 5. Distribution of Order Spend and Customer Purchase Frequencies
Analyzing basket economics and customer purchase velocities.

In [6]:
fig5_path = plot_order_value_and_basket_distributions(df_clean, output_dir=os.path.join(project_root, "reports", "figures"))
print(f"Figure saved to: {os.path.relpath(fig5_path, project_root)}")

spend_stats = df_clean[["Revenue", "Previous_Purchases"]].describe().T
spend_stats

Figure saved to: reports/figures/05_order_value_distribution.png


![Distribution of Order Spend & Purchases](../reports/figures/05_order_value_distribution.png)

> **Business Interpretation:** Order transaction amounts are uniformly spread between $20 and $100 with an average spend of $59.76, while a high mean count of 25.4 prior purchases demonstrates strong customer lifetime retention that can be further monetized via loyalty tier incentives.

### 6. Merchandise Categories & Description Keywords (Text-Derived)
Evaluating high-level category contributions and analyzing token frequencies parsed directly from product text descriptions.

In [7]:
fig6_path = plot_category_and_keyword_analysis(df_clean, output_dir=os.path.join(project_root, "reports", "figures"))
print(f"Figure saved to: {os.path.relpath(fig6_path, project_root)}")

cat_summary = df_clean.groupby("Category").agg(
    Revenue=("Revenue", "sum"),
    Orders=("CustomerID", "count")
).sort_values(by="Revenue", ascending=False)
cat_summary["Share (%)"] = (cat_summary["Revenue"] / cat_summary["Revenue"].sum() * 100).round(2)
cat_summary

Figure saved to: reports/figures/06_category_keyword_analysis.png


![Category and Keyword Analysis](../reports/figures/06_category_keyword_analysis.png)

> **Business Interpretation:** Clothing ($104.3K; 44.7%) and Accessories ($74.2K; 31.8%) comprise over 76% of total revenue, indicating strong opportunities to bundle high-margin Footwear and Outerwear items into apparel checkouts to expand average order size.

### Key findings

1. **Steady Year-Round Revenue Flow**: Sales maintain a steady baseline across all seasons (~$58,000 per season) with total revenue of **$233,081.00**, demonstrating resilient non-seasonal customer demand.
2. **Operational Peak Windows**: Order frequency peaks mid-week and between 11:00 AM and 2:00 PM, defining the highest-converting windows for targeted promotional campaigns.
3. **Hero Merchandise & Revenue Engines**: The top 4 revenue drivers—Blouse ($10,410), Shirt ($10,332), Dress ($10,320), and Pants ($10,090)—consistently outperform other items and should be prioritized for catalog promotions.
4. **Category Concentration**: Clothing (44.7%) and Accessories (31.8%) represent **76.5% of total business revenue**, while Footwear (15.5%) and Outerwear (8.0%) offer substantial untapped cross-selling potential.
5. **National Market Diversification**: Transactions span all 50 US States without disproportionate regional concentration (top 5 states comprise only 12.5% of total sales: Montana, California, Idaho, Illinois, Alabama), reducing single-market dependency risk.
6. **Strong Customer Retention Base**: Shoppers have completed an average of **25.4 historical purchases**, providing a solid foundation for customer tiering and personalized RFM clustering in Phase 4.